# 实例分割

## 介绍

语义分割为每个像素分配类别标签，但无法区分同类的单个对象。实例分割通过单独检测每个对象并为每个实例生成单独的掩码来解决这个问题。

这种区分对于许多地理空间应用至关重要。农业分析师需要单个田地地块进行作物监测和产量估算，城市规划师需要计算建筑物数量并测量占地面积，保护组织使用树冠描绘来监测森林健康。

随着卫星图像可用性的增长，自动实例分割变得越来越重要。像欧盟共同农业政策这样的计划需要数百万地块的准确每块田信息，使得手动数字化不切实际。

本教程介绍用于遥感实例分割的Mask R-CNN。您将下载[Fields of The World](https://fieldsofthe.world)基准数据集，准备Sentinel-2图像，使用`geoai`训练Mask R-CNN模型，运行推理，清理和向量化预测，并提取每个检测田地的几何属性。

## 学习目标

在本教程结束时，您将能够：

- 解释实例分割与语义分割的区别以及每种方法何时适用
- 描述Mask R-CNN架构及其关键组件（骨干网络、RPN、检测头、掩码头）
- 下载并探索用于农田边界检测的Fields of The World基准数据集
- 准备Sentinel-2多光谱图像和实例分割掩码用于模型训练
- 使用`geoai`包训练用于田地边界描绘的Mask R-CNN模型
- 在测试图像上运行滑动窗口推理并解释原始预测掩码
- 通过移除小的虚假检测和填充孔洞来清理实例掩码
- 将栅格预测向量化为多边形特征并提取几何属性
- 批量处理多个图像以进行高效的大规模推理

## 实例分割 vs. Semantic Segmentation

**语义分割**生成单个标签图，其中每个像素属于一个类别。相邻田地都获得相同的"田地"标签，它们之间没有边界，这足以映射总类别范围，但不足以识别单个地块。

**实例分割**为每个检测到的对象生成单独的掩码，具有自己的置信度分数和唯一标识符。这支持计数、每个对象的测量和跨时间跟踪，但代价是模型复杂度更高。

例如，应用于农业区域的语义分割模型生成单个连接的"农田"区域，无法确定单个田地的数量。实例分割模型为每个田地生成单独的掩码，可以转换为具有面积、周长和伸长率等几何属性的多边形。

实例分割和语义分割是互补的。一些架构将两者结合成全景分割，但对于大多数地理空间工作流程，特定特征（田地、建筑物、树木）的实例分割是主要用例。

## Mask R-CNN架构

Mask R-CNN通过添加掩码预测分支扩展了Faster R-CNN目标检测框架。该架构有四个主要组件。

### 骨干编码器

骨干网络是一个CNN（通常是ResNet-50或ResNet-101），结合特征金字塔网络（FPN），在多个尺度上提取分层特征图。对于具有四个波段（红、绿、蓝、近红外）的Sentinel-2图像，第一个卷积层被调整为接受四个输入通道。

### 区域候选网络

RPN扫描特征图并提出可能包含对象的矩形区域。它生成目标性分数和边界框调整，然后通过非极大值抑制过滤候选区域以去除冗余的重叠框。

### 检测头

每个候选区域使用RoI Align从特征图中采样。检测头将每个区域分类为对象类别（或背景）并细化边界框坐标。

### 掩码头

对于每个检测到的对象，一个小型全卷积网络在边界框区域内预测二值掩码。掩码头通过单独的分支操作，因此掩码预测不会干扰类别或框预测。输出是固定大小的掩码（通常为28×28像素），调整大小以匹配检测到的边界框。

### RoI Align：保持空间精度

RoI Align使用双线性插值在精确的所需位置计算特征值，无需舍入，消除了旧版RoI Pooling的量化误差。这种精度对于像农田这样的小物体至关重要，它们在训练芯片中可能只占用几十个像素。

### 完整管道

骨干网络提取多尺度特征，RPN提出候选区域，检测头预测类别标签和边界框，掩码头为每个实例预测二值掩码。经过非极大值抑制后，输出是一组实例，每个实例具有类别标签、置信度分数、边界框和像素级掩码。

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install "geoai-py[extra]"

## 下载FTW数据集

[Fields of The World](https://fieldsofthe.world)数据集是农田边界检测的大规模基准数据集。它包含来自24个国家的70,462个图像芯片，将Sentinel-2图像（10米分辨率的四个波段）与实例分割掩码配对。该数据集可在[Source Cooperative](https://source.coop/kerner-lab/fields-of-the-world)上获取。

我们使用[卢森堡子集](https://source.coop/kerner-lab/fields-of-the-world/luxembourg)，这是最小的国家子集之一，非常适合教程。

In [ ]:
from pathlib import Path

import geopandas as gpd
import geoai

In [ ]:
geoai.download_ftw(countries=["luxembourg"], output_dir="ftw_data")

### 探索数据集

FTW数据集包含一个GeoParquet文件，包含每个芯片的元数据和几何信息，包括官方的训练、验证和测试分割。

In [ ]:
country_dir = Path("ftw_data") / "luxembourg"
chips_gdf = gpd.read_parquet(country_dir / "chips_luxembourg.parquet")

print(f"Total chips: {len(chips_gdf)}")
print(f"\nSplit distribution:")
print(chips_gdf["split"].value_counts())

可视化卢森堡境内训练、验证和测试芯片的空间分布。

In [ ]:
geoai.view_vector_interactive(chips_gdf, column="split")

显示示例图像-掩码对。每个掩码使用唯一的整数ID来区分各个田地实例。

In [ ]:
geoai.display_ftw_samples("ftw_data", country="luxembourg", num_samples=4)

## 准备训练数据

`geoai` Mask R-CNN管道期望`images/`和`labels/`目录包含uint8 GeoTIFF文件。`prepare_ftw`函数将原始Sentinel-2反射率值（0-10,000）重新缩放到0-255，将文件组织成预期结构，并留出测试芯片用于推理。

In [ ]:
data = geoai.prepare_ftw("ftw_data", country="luxembourg")
data

```text
FTW卢森堡：共808个芯片
  训练：643，验证：81，测试：84
  使用724个芯片进行训练
准备训练数据...
准备了724个训练芯片（跳过0个）
准备了5个测试芯片
```

该函数将643个训练芯片和81个验证芯片合并为724个芯片用于模型开发。在训练期间，使用`val_split=0.2`创建新的验证分割。

通过显示一些图像-标签对来验证准备好的瓦片。

In [ ]:
geoai.display_training_tiles(
    output_dir="field_boundaries",
    num_tiles=4,
    figsize=(12, 6),
    cmap="tab20",
)

## 训练Mask R-CNN模型

我们训练一个带有ResNet-50 + FPN骨干的Mask R-CNN模型。关键参数：

- **`num_classes=2`**：背景（0）和田地（1）。所有田地都属于单个类别；实例通过单独的掩码区分。
- **`num_channels=4`**：Sentinel-2波段（红、绿、蓝、近红外）。近红外波段有助于区分RGB中不可见的植被边界。
- **`instance_labels=True`**：FTW掩码已经为每个田地编码了唯一的实例ID，因此`geoai`直接使用它们。
- **`num_epochs=20`**：对于教程足够；生产环境增加到50-100。
- **`val_split=0.2`**: Creates an internal validation split for monitoring performance during training.

In [ ]:
geoai.train_instance_segmentation_model(
    images_dir=data["images_dir"],
    labels_dir=data["labels_dir"],
    output_dir="field_boundaries/models",
    num_classes=2,
    num_channels=4,
    batch_size=4,
    num_epochs=20,
    learning_rate=0.005,
    val_split=0.2,
    instance_labels=True,
    visualize=True,
    verbose=True,
)

总损失是RPN目标性损失、分类损失、边界框回归损失和掩码损失的加权和。持续下降的损失表示训练正常。如果遇到内存不足错误，将批量大小减少到2或使用更轻量的骨干网络。

绘制训练指标以评估收敛情况。

In [ ]:
geoai.plot_performance_metrics(
    history_path="field_boundaries/models/training_history.pth",
    figsize=(15, 5),
    verbose=True,
)

## 运行推理

使用滑动窗口推理将训练好的模型应用于测试图像。128像素的重叠防止瓦片边界附近的对象被分割。设置`vectorize=True`时，函数返回实例掩码、类别标签、置信度分数和向量化多边形。

In [ ]:
test_images = sorted(Path(data["test_dir"]).glob("*.tif"))
test_image_path = str(test_images[0])
masks_path = "field_boundary_prediction.tif"
model_path = "field_boundaries/models/best_model.pth"

result = geoai.instance_segmentation(
    input_path=test_image_path,
    output_path=masks_path,
    model_path=model_path,
    num_classes=2,
    num_channels=4,
    window_size=256,
    overlap=128,
    confidence_threshold=0.5,
    batch_size=4,
    vectorize=True,
    class_names=["background", "building"],
)
result

### 可视化原始预测

结果字典包含四个键：`"instance"`（每个田地的唯一整数ID）、`"class_label"`（预测类别）、`"score"`（置信度）和`"vector"`（每个田地一个多边形的GeoDataFrame）。

可视化实例掩码，其中每种颜色代表一个不同的检测田地。

In [ ]:
geoai.view_raster(
    result["instance"],
    nodata=0,
    cmap="tab20",
    basemap=test_image_path,
    backend="ipyleaflet",
)

可视化类别标签栅格和置信度分数栅格。

In [ ]:
geoai.view_raster(
    result["class_label"],
    nodata=0,
    cmap="binary",
    basemap=test_image_path,
    backend="ipyleaflet",
)

In [ ]:
geoai.view_raster(
    result["score"], nodata=0, basemap=test_image_path, backend="ipyleaflet"
)

显示按置信度分数着色的向量化预测。

In [ ]:
geoai.view_vector_interactive(result["vector"], tiles=test_image_path, column="score")

## 后处理预测

原始实例掩码通常包含小的虚假检测或孔洞。`clean_instance_mask`函数移除小于最小面积的连通组件并填充指定大小的孔洞。

In [ ]:
cleaned_masks_path = "field_boundary_prediction_cleaned.tif"
geoai.clean_instance_mask(
    result["instance"], cleaned_masks_path, min_area=100, max_hole_area=100
)

In [ ]:
geoai.view_raster(
    cleaned_masks_path,
    nodata=0,
    cmap="tab20",
    basemap=test_image_path,
    backend="ipyleaflet",
)

### 向量化预测

将清理后的栅格掩码转换为矢量多边形。每个唯一的像素值成为地理参考GeoDataFrame中的单独多边形特征。

In [ ]:
output_vector_path = "field_boundary_prediction.geojson"
gdf = geoai.raster_to_vector(cleaned_masks_path, output_vector_path)

### 比较预测与图像

使用分割地图直观比较检测到的田地边界与原始Sentinel-2图像。

In [ ]:
geoai.create_split_map(
    left_layer=gdf,
    right_layer=test_image_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_image_path,
)

## 提取几何属性

计算几何属性将原始预测转换为丰富的属性表用于空间分析。如果使用经纬度坐标，请先投影到投影坐标系。

| Property       | Description                                                                |
| -------------- | -------------------------------------------------------------------------- |
| **面积**       | 田地大小（公顷），对产量估算和补贴计划至关重要                |
| **Perimeter**  | Boundary length in meters, useful for fencing cost estimation              |
| **Elongation** | Major/minor axis ratio, distinguishes strip fields from compact parcels    |
| **Solidity**   | Area/convex hull area ratio, measures boundary irregularity                |
| **Extent**     | Area/bounding box area ratio, indicates how rectangular a field is         |

In [ ]:
gdf_props = geoai.add_geometric_properties(gdf, area_unit="ha", length_unit="m")
gdf_props.head()

In [ ]:
gdf_props.describe()

### 按属性可视化田地

按几何属性着色的交互式地图揭示田地特征的空间模式。

In [ ]:
geoai.view_vector_interactive(gdf_props, column="area_ha", tiles=test_image_path)

伸长率突出显示条形田地与紧凑地块的对比。

In [ ]:
geoai.view_vector_interactive(gdf_props, column="elongation", tiles=test_image_path)

## 批量处理

批量处理将同一模型应用于目录中的所有图像，避免重复加载模型并更好地利用GPU资源。

In [ ]:
geoai.instance_segmentation_batch(
    input_dir=data["test_dir"],
    output_dir="field_boundaries/predictions",
    model_path=model_path,
    num_classes=2,
    num_channels=4,
    window_size=256,
    overlap=128,
    confidence_threshold=0.5,
    batch_size=4,
)

## 关键要点

1. 实例分割使用单独的掩码检测单个对象，支持计数和每个对象的测量，这与语义分割不同。

2. Mask R-CNN结合骨干编码器、区域候选网络、检测头和掩码头来分类、定位和分割每个对象实例。

3. Fields of The World数据集将Sentinel-2图像与24个国家的实例掩码配对，用于可重现的田地边界检测。

4. 近红外波段捕获RGB中不可见的植被差异，改善相邻田地之间的边界检测。

5. `instance_labels=True`标志告诉管道直接使用掩码中预先存在的唯一实例ID。

6. 使用`clean_instance_mask`进行后处理可移除小的虚假检测并填充孔洞以获得更干净的输出。

7. `raster_to_vector`函数将每个唯一实例ID转换为单独的GIS就绪多边形特征。

8. 几何属性（面积、周长、伸长率、坚实度、范围）支持研究区域内田地特征的统计分析。

9. Batch inference processes an entire directory of images in one call, avoiding repeated model loading.